# FXS / 10b — What separates winning days from losing days? (2020–2025, one look at 2026)

Pre-registration: `discussions/FX_Intraday_Seasonality/006_claude_fxs10b_winning_days_preregistration.md`.

For each of the 12 signals, every day of 2020–2025 is labelled **winner** or **loser** by the mid-price drift between entry and exit (in the trade direction). The stacked-day scan then compares winners with losers on every interval:
- in the **24 h before entry** (a predictor: trade only when the pre-entry move points to a winner);
- in the **24 h after exit** (a second trade, opened once the first trade's result is known).

The trade window itself is not scanned: winners rise there by definition.

Tests per signal: winner − loser difference (Welch t), Benjamini–Hochberg over the signal's 2,704 tests (E1), same sign in ≥ 5 of 6 years (E2), and 20 label-shuffle placebos (E0). Candidates are frozen, then checked once on 2026. Results in **net pips** ($3/lot, 0.9 × spread).

In [1]:
import hashlib, json, os, sys, tempfile
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

COMMISSION_PER_LOT = 3.0
SPREAD_FACTOR = 0.9
N_PLACEBO = 20
Q = 0.05
MIN_YEARS = 5                      # of 6
MAX_CAND = 10
WINDOWS = ((dt.date(2020, 1, 1), dt.date(2020, 12, 31), "discovery"), (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"))
HOLDOUT = (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout")
DURS = (15, 30, 60, 120, 240)
SPAN = 1440                        # minutes scanned before entry and after exit
SIGNALS = {
    "EURUSD-02": ("EURUSD", "NY", 660, 900, -1), "USDJPY-01": ("USDJPY", "UTC", 55, 115, -1), "USDJPY-02": ("USDJPY", "UTC", 25, 55, 1),
    "N-EU-1": ("EURUSD", "NY", 885, 900, -1), "N-UJ-1": ("USDJPY", "UTC", 55, 70, -1), "N-UJ-3": ("USDJPY", "NY", 1340, 1400, 1),
    "N-EJ-1": ("EURJPY", "NY", 1315, 1435, 1), "N-EJ-2": ("EURJPY", "NY", 780, 900, -1),
    "AUDJPY-N1": ("AUDJPY", "NY", 1195, 1435, 1), "AUDJPY-N2": ("AUDJPY", "UTC", 60, 180, 1), "CADJPY-N1": ("CADJPY", "NY", 1070, 1310, 1),
    "GBPJPY-N1": ("GBPJPY", "NY", 1195, 1435, 1)}

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs10b_")
    os.environ.pop("DUKASCOPY_DIR", None)
    N_PLACEBO = 3
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
OUT = Path.cwd() / ("10b_synthetic" if SYNTHETIC else "10b"); OUT.mkdir(exist_ok=True)
PAIRS = sorted({v[0] for v in SIGNALS.values()})
pd.set_option("display.width", 240, "display.max_columns", 40, "display.max_rows", 300)
np.seterr(invalid="ignore", divide="ignore")

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

## 0. Synthetic inputs (smoke run only)

In [2]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(PAIRS):
        F.write_synthetic_candles(base, p, dt.date(2019, 12, 28), dt.date(2026, 11, 6), seed=71 + i, drift_bps=-3.0 if p == "EURUSD" else 0.0,
                                  drift_ny=(4, 8), price=150.0 if p.endswith("JPY") else 1.10)

## 1. Price paths around every trade

For each signal and day: bid and ask at every minute from 24 h before entry to 24 h after exit (price at an instant = close of the minute before it, as in the engine). Minutes while the market is closed (Fri 17:00 → Sun 17:00 New York) are missing.

In [3]:
def minutes(pair, windows):
    m = pd.concat([F.load_minutes(pair, a, b, st, min_file_share=cfg.min_file_share) for a, b, st in windows])
    m = m[~m.index.duplicated(keep="last")].sort_index()
    assert (m.index[-1] - m.index[0]) // pd.Timedelta(minutes=1) + 1 == len(m), f"{pair}: minute index has gaps"
    ny = m.index.tz_convert(F.NY)
    wd, hr = ny.weekday, ny.hour
    closed = ((wd == 4) & (hr >= 17)) | (wd == 5) | ((wd == 6) & (hr < 17))
    m.loc[closed, ["bid", "ask"]] = np.nan
    return m

def paths(sig, m, start, end):
    """Rows = signal days; columns = minutes relative to entry, -SPAN .. dur + SPAN. Returns dates, bid, ask, dur."""
    pair, clock, s, e, d = SIGNALS[sig]
    dur = e - s
    days = pd.bdate_range(start, end)
    if clock == "NY":
        entry = pd.DatetimeIndex([(pd.Timestamp(x) - pd.Timedelta(days=1) + pd.Timedelta(hours=17)).tz_localize(F.NY) for x in days]).tz_convert("UTC")
    else:
        entry = days.tz_localize("UTC")
    entry = entry + pd.Timedelta(minutes=s)
    t0 = m.index[0]
    rel = np.arange(-SPAN, dur + SPAN + 1)
    pos = ((entry - t0) // pd.Timedelta(minutes=1)).to_numpy()[:, None] + rel[None, :] - 1     # close of the minute before
    ok = (pos >= 0) & (pos < len(m))
    pc = np.clip(pos, 0, len(m) - 1)
    bid, ask = m["bid"].to_numpy()[pc], m["ask"].to_numpy()[pc]
    bid[~ok], ask[~ok] = np.nan, np.nan
    mid, half = 0.5 * (bid + ask), 0.5 * (ask - bid) * SPREAD_FACTOR
    return pd.DatetimeIndex(days), mid - half, mid + half, dur

def interval_grid(dur):
    """(zone, start, end) in minutes relative to entry, starts on a 5-minute grid."""
    rows = []
    for L in DURS:
        rows += [("pre", a, a + L) for a in range(-SPAN, -L + 1, 5)]
        rows += [("post", a, a + L) for a in range(dur, dur + SPAN - L + 1, 5)]
    return pd.DataFrame(rows, columns=["zone", "start", "end"])

M = {p: minutes(p, WINDOWS) for p in PAIRS}
DATA = {}
for sig in SIGNALS:
    days, bid, ask, dur = paths(sig, M[SIGNALS[sig][0]], WINDOWS[0][0], WINDOWS[-1][1])
    lm = np.log(0.5 * (bid + ask))
    c0 = SPAN
    drift = (lm[:, c0 + dur] - lm[:, c0]) * 1e4 * SIGNALS[sig][4]
    keep = np.isfinite(drift) & (drift != 0)
    DATA[sig] = dict(days=days[keep], bid=bid[keep], ask=ask[keep], lm=lm[keep], dur=dur, win=drift[keep] > 0, drift=drift[keep])
summ = pd.DataFrame({s: {"days": len(v["days"]), "winners": int(v["win"].sum()), "win rate": v["win"].mean(),
                         "mean drift winners (bps)": v["drift"][v["win"]].mean(), "mean drift losers (bps)": v["drift"][~v["win"]].mean()}
                     for s, v in DATA.items()}).T
display(summ.round(3))

C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: DatetimeIndex.weekday is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  wd, hr = ny.weekday, ny.hour
C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: DatetimeIndex.weekday is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  wd, hr = ny.weekday, ny.hour
C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: DatetimeIndex.weekday is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  wd, hr = ny.weekday, ny.hour
C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: DatetimeIndex.weekday is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  wd, hr = ny.weekday, ny.hour
C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: Datetim

,days,winners,win rate,mean drift winners (bps),mean drift losers (bps)
EURUSD-02,1555.0,848.0,0.545,15.187,-14.185
USDJPY-01,1560.0,888.0,0.569,9.499,-8.582
USDJPY-02,1558.0,876.0,0.562,7.171,-6.307
N-EU-1,1554.0,879.0,0.566,4.541,-4.093
N-UJ-1,1554.0,879.0,0.566,5.551,-4.692
N-UJ-3,1549.0,870.0,0.562,4.601,-3.844
N-EJ-1,1555.0,906.0,0.583,6.755,-5.797
N-EJ-2,1558.0,799.0,0.513,12.031,-10.524
AUDJPY-N1,1556.0,923.0,0.593,14.646,-15.864
AUDJPY-N2,1558.0,832.0,0.534,17.585,-16.753


## 2. Scan: winner − loser difference on every interval

In [4]:
def diff_stats(R, w):
    """Column-wise Welch test of mean(R | w) - mean(R | ~w), ignoring NaN."""
    v = np.isfinite(R)
    R0 = np.where(v, R, 0.0)
    out = {}
    for k, g in (("w", w), ("l", ~w)):
        mk = v & g[:, None]
        n = mk.sum(0)
        mu = (R0 * mk).sum(0) / n
        var = (((R0 - mu) ** 2) * mk).sum(0) / (n - 1)
        out[k] = (n, mu, var)
    (n1, m1, v1), (n0, m0, v0) = out["w"], out["l"]
    t = (m1 - m0) / np.sqrt(v1 / n1 + v0 / n0)
    return m1 - m0, t, 2 * stats.norm.sf(np.abs(t)), m1, m0

def scan(sig, w, years):
    D = DATA[sig]
    iv = interval_grid(D["dur"])
    R = (D["lm"][:, iv["end"].to_numpy() + SPAN] - D["lm"][:, iv["start"].to_numpy() + SPAN]) * 1e4
    diff, t, p, m1, m0 = diff_stats(R, w)
    same = np.zeros(len(iv), int)
    for y in np.unique(years):
        sel = years == y
        same += (np.sign(diff_stats(R[sel], w[sel])[0]) == np.sign(diff)).astype(int)
    rej, q = F.bh_reject(p, Q)
    return iv.assign(diff_bps=diff, t=t, p=p, q=q, winners_bps=m1, losers_bps=m0, years_same_sign=same,
                     E1=rej, E2=same >= MIN_YEARS), R

SCANS, RS, E0 = {}, {}, []
rng = np.random.default_rng(10)
for sig, D in DATA.items():
    years = D["days"].year.to_numpy()
    sc, R = scan(sig, D["win"], years)
    SCANS[sig], RS[sig] = sc, R
    real = int((sc["E1"] & sc["E2"]).sum())
    pl = []
    for _ in range(N_PLACEBO):
        w = D["win"].copy()
        for y in np.unique(years):
            ix = np.where(years == y)[0]
            w[ix] = rng.permutation(w[ix])
        ps, _ = scan(sig, w, years)
        pl.append(int((ps["E1"] & ps["E2"]).sum()))
    p95 = float(np.percentile(pl, 95))
    E0.append({"signal": sig, "E1": int(sc["E1"].sum()), "E1+E2": real, "E1+E2 pre": int((sc["E1"] & sc["E2"] & (sc["zone"] == "pre")).sum()),
               "E1+E2 post": int((sc["E1"] & sc["E2"] & (sc["zone"] == "post")).sum()), "placebo mean": np.mean(pl), "placebo p95": p95,
               "E0": "STRUCTURE_FOUND" if real > p95 else "NO_STRUCTURE"})
    print(f"{sig}: E1 {int(sc['E1'].sum())}, E1+E2 {real}, placebo p95 {p95:.1f}")
E0 = pd.DataFrame(E0).set_index("signal")
display(E0); E0.to_csv(OUT / "10b_E0.csv")

EURUSD-02: E1 0, E1+E2 0, placebo p95 0.1
USDJPY-01: E1 0, E1+E2 0, placebo p95 0.0
USDJPY-02: E1 0, E1+E2 0, placebo p95 0.4
N-EU-1: E1 0, E1+E2 0, placebo p95 0.0
N-UJ-1: E1 0, E1+E2 0, placebo p95 0.0
N-UJ-3: E1 7, E1+E2 7, placebo p95 0.0
N-EJ-1: E1 4, E1+E2 4, placebo p95 0.0
N-EJ-2: E1 0, E1+E2 0, placebo p95 0.1
AUDJPY-N1: E1 1, E1+E2 1, placebo p95 0.0
AUDJPY-N2: E1 0, E1+E2 0, placebo p95 0.2
CADJPY-N1: E1 0, E1+E2 0, placebo p95 0.0
GBPJPY-N1: E1 5, E1+E2 5, placebo p95 0.0


,E1,E1+E2,E1+E2 pre,E1+E2 post,placebo mean,placebo p95,E0
signal,,,,,,,
EURUSD-02,0,0,0,0,0.10,0.10,NO_STRUCTURE
USDJPY-01,0,0,0,0,0.00,0.00,NO_STRUCTURE
USDJPY-02,0,0,0,0,0.35,0.35,NO_STRUCTURE
N-EU-1,0,0,0,0,0.00,0.00,NO_STRUCTURE
N-UJ-1,0,0,0,0,0.00,0.00,NO_STRUCTURE
N-UJ-3,7,7,0,7,0.00,0.00,STRUCTURE_FOUND
N-EJ-1,4,4,0,4,0.00,0.00,STRUCTURE_FOUND
N-EJ-2,0,0,0,0,0.05,0.05,NO_STRUCTURE
AUDJPY-N1,1,1,0,1,0.00,0.00,STRUCTURE_FOUND


## 3. Candidates (E1 + E2, strongest non-overlapping, ≤ 10 per signal) and the frozen file

Times are shown relative to entry (pre) or exit (post) and as clock times.

In [5]:
def clock_time(sig, rel):
    pair, clock, s, e, d = SIGNALS[sig]
    return F.slot_label(clock, int((s + rel) % 1440))

rows = []
for sig, sc in SCANS.items():
    surv = sc[sc["E1"] & sc["E2"]].reindex(sc.loc[sc["E1"] & sc["E2"], "t"].abs().sort_values(ascending=False).index)
    kept = []
    for _, r in surv.iterrows():
        if any(k["zone"] == r["zone"] and r["start"] < k["end"] and k["start"] < r["end"] for k in kept):
            continue
        kept.append(r)
        if len(kept) == MAX_CAND:
            break
    for k, r in enumerate(kept):
        rows.append({"id": f"{sig}/{r['zone']}{k + 1}", "signal": sig, "zone": r["zone"], "start": int(r["start"]), "end": int(r["end"]),
                     "minutes": int(r["end"] - r["start"]), "from": clock_time(sig, r["start"]), "to": clock_time(sig, r["end"]).split()[1],
                     "rel": (f"entry{r['start']:+d} -> entry{r['end']:+d} min" if r["zone"] == "pre" else
                             f"exit+{r['start'] - DATA[sig]['dur']} -> exit+{r['end'] - DATA[sig]['dur']} min"),
                     "sign": int(np.sign(r["diff_bps"])), "diff_bps": r["diff_bps"], "winners_bps": r["winners_bps"], "losers_bps": r["losers_bps"],
                     "t": r["t"], "q": r["q"], "years_same_sign": int(r["years_same_sign"])})
cand = pd.DataFrame(rows)
cols = ["id", "signal", "zone", "start", "end", "minutes", "from", "to", "rel", "sign", "diff_bps", "winners_bps", "losers_bps", "t", "q", "years_same_sign"]
cand = cand.reindex(columns=cols)
ff = OUT / "10b_candidates_frozen.csv"
cand.to_csv(ff, index=False, lineterminator="\n")
SHA = hashlib.sha256(ff.read_bytes()).hexdigest()
print(f"{len(cand)} candidates frozen, sha256 {SHA}")
display(cand.round(3))

4 candidates frozen, sha256 3bce9b89b52e41043c9556c8ebaadbbfdfed3c56b35b4dc3687b0528a10f45ff


,id,signal,zone,start,end,minutes,from,to,rel,sign,diff_bps,winners_bps,losers_bps,t,q,years_same_sign
0,N-UJ-3/post1,N-UJ-3,post,95,110,15,NY 16:55,17:10,exit+35 -> exit+50 min,-1,-1.163,-1.361,-0.198,-6.566,0.000,6
1,N-EJ-1/post1,N-EJ-1,post,120,135,15,NY 16:55,17:10,exit+0 -> exit+15 min,-1,-1.261,-1.396,-0.135,-6.731,0.000,6
2,AUDJPY-N1/post1,AUDJPY-N1,post,240,255,15,NY 16:55,17:10,exit+0 -> exit+15 min,-1,-1.343,-1.893,-0.550,-4.855,0.003,5
3,GBPJPY-N1/post1,GBPJPY-N1,post,240,255,15,NY 16:55,17:10,exit+0 -> exit+15 min,-1,-2.116,-1.985,0.132,-7.663,0.000,6


## 4. Trading measures, 2020–2025 (in sample)

- **pre:** the signal's net pips on days the pre-entry move has the winners' sign (`agree`) against days it does not; win rate in each group.
- **post:** follow-on trade on the interval: in the candidate's sign after a winning day, opposite after a losing day. Net pips per trade.

In [6]:
def comm_pips(pair, price):
    return F.bps_to_pips(F.commission_bps_per_lot(pair, COMMISSION_PER_LOT), price, pair)

def net_pips(D, pair, a, b, direction):
    """Per-day net pips of a trade from relative minute a to b; direction per day (+1/-1) or scalar."""
    bid, ask = D["bid"][:, a + SPAN], D["ask"][:, a + SPAN]
    bid2, ask2 = D["bid"][:, b + SPAN], D["ask"][:, b + SPAN]
    direction = np.broadcast_to(direction, bid.shape)
    gross = np.where(direction > 0, np.log(bid2 / ask), np.log(bid / ask2)) * 1e4
    price = 0.5 * (bid + ask)
    return F.bps_to_pips(gross, price, pair) - comm_pips(pair, price)

def measures(c, D):
    pair, clock, s, e, d = SIGNALS[c["signal"]]
    R = (D["lm"][:, c["end"] + SPAN] - D["lm"][:, c["start"] + SPAN]) * 1e4
    sig_net = net_pips(D, pair, 0, D["dur"], d)
    yrs = D["days"].year
    if c["zone"] == "pre":
        ag = np.sign(R) == c["sign"]
        ok = np.isfinite(R) & np.isfinite(sig_net)
        x_ag, x_dis = sig_net[ok & ag], sig_net[ok & ~ag]
        out = {"rule": "trade the signal only on agree days", "net pips (rule)": x_ag.mean(), "net pips (other days)": x_dis.mean(),
               "net pips (always)": sig_net[ok].mean(), "win rate agree": D["win"][ok & ag].mean(), "win rate other": D["win"][ok & ~ag].mean(),
               "share of days traded": (ok & ag).sum() / ok.sum(), "t (rule)": x_ag.mean() / (x_ag.std() / np.sqrt(len(x_ag)))}
        per_day = np.where(ok & ag, sig_net, np.nan)
    else:
        direction = np.where(D["win"], c["sign"], -c["sign"])
        fo = net_pips(D, pair, c["start"], c["end"], direction)
        ok = np.isfinite(fo)
        out = {"rule": "follow-on trade after exit", "net pips (rule)": fo[ok].mean(), "t (rule)": fo[ok].mean() / (fo[ok].std() / np.sqrt(ok.sum())),
               "after winners": fo[ok & D["win"]].mean(), "after losers": fo[ok & ~D["win"]].mean(), "trades": int(ok.sum())}
        per_day = fo
    yr = pd.Series(per_day, index=yrs).groupby(level=0).mean()
    return out, yr

MEAS, YR = [], []
for _, c in cand.iterrows():
    o, yr = measures(c, DATA[c["signal"]])
    MEAS.append({"id": c["id"], **o}); YR.append(yr.rename(c["id"]))
MEAS = pd.DataFrame(MEAS).set_index("id") if MEAS else pd.DataFrame()
YR = pd.DataFrame(YR) if YR else pd.DataFrame()
if len(MEAS):
    display(MEAS.round(3)); MEAS.to_csv(OUT / "10b_measures_2020_2025.csv")
    print("rule net pips per trade by year:"); display(YR.round(2)); YR.to_csv(OUT / "10b_measures_by_year.csv")
else:
    print("no candidates")

,rule,net pips (rule),t (rule),after winners,after losers,trades
id,,,,,,
N-UJ-3/post1,follow-on trade after exit,-2.319,-17.163,-1.341,-3.567,1239
N-EJ-1/post1,follow-on trade after exit,-4.486,-27.991,-3.629,-5.700,1245
AUDJPY-N1/post1,follow-on trade after exit,-4.271,-30.359,-3.430,-5.536,1245
GBPJPY-N1/post1,follow-on trade after exit,-7.451,-29.433,-6.288,-9.129,1246


rule net pips per trade by year:


,2020,2021,2022,2023,2024,2025
N-UJ-3/post1,-1.53,-1.23,-2.49,-2.35,-2.38,-3.95
N-EJ-1/post1,-2.15,-2.52,-4.36,-5.46,-5.68,-6.76
AUDJPY-N1/post1,-2.44,-2.14,-4.72,-5.28,-4.78,-6.28
GBPJPY-N1/post1,-4.90,-3.56,-7.99,-8.24,-9.09,-10.94


## 5. 2026, one look

2026 minutes are loaded only now, after the candidates are frozen. A candidate **holds** if its winner − loser difference has the same sign in 2026 and its trading measure is positive.

In [7]:
assert hashlib.sha256(ff.read_bytes()).hexdigest() == SHA
H = []
if len(cand):
    M26 = {p: minutes(p, (HOLDOUT,)) for p in sorted({SIGNALS[s][0] for s in cand["signal"]})}
    for sig in sorted(set(cand["signal"])):
        days, bid, ask, dur = paths(sig, M26[SIGNALS[sig][0]], *HOLDOUT[:2])
        lm = np.log(0.5 * (bid + ask))
        drift = (lm[:, SPAN + dur] - lm[:, SPAN]) * 1e4 * SIGNALS[sig][4]
        keep = np.isfinite(drift) & (drift != 0)
        D26 = dict(days=days[keep], bid=bid[keep], ask=ask[keep], lm=lm[keep], dur=dur, win=drift[keep] > 0, drift=drift[keep])
        for _, c in cand[cand["signal"] == sig].iterrows():
            R = (D26["lm"][:, c["end"] + SPAN] - D26["lm"][:, c["start"] + SPAN]) * 1e4
            dif, t, p, _, _ = diff_stats(R[:, None], D26["win"])
            o, _ = measures(c, D26)
            H.append({"id": c["id"], "2020-25 diff bps": c["diff_bps"], "2026 diff bps": dif[0], "2026 t": t[0], "same sign": np.sign(dif[0]) == c["sign"],
                      "2026 net pips (rule)": o["net pips (rule)"], "2026 t (rule)": o["t (rule)"],
                      "2026 net pips (always)": o.get("net pips (always)", np.nan),
                      "holds": bool(np.sign(dif[0]) == c["sign"] and o["net pips (rule)"] > 0)})
H = pd.DataFrame(H)
if len(H):
    display(H.set_index("id").round(3)); H.to_csv(OUT / "10b_holdout_2026.csv", index=False)
    print(f"holds in 2026: {int(H['holds'].sum())} of {len(H)}  (by chance about half would show the same sign)")

C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: DatetimeIndex.weekday is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  wd, hr = ny.weekday, ny.hour
C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: DatetimeIndex.weekday is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  wd, hr = ny.weekday, ny.hour
C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: DatetimeIndex.weekday is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  wd, hr = ny.weekday, ny.hour
C:\Users\Iván\AppData\Local\Temp\ipykernel_8564\3092481437.py:6: Pandas4Warning: DatetimeIndex.weekday is deprecated and will be removed in a future version. Use DatetimeIndex.day_of_week instead.
  wd, hr = ny.weekday, ny.hour


,2020-25 diff bps,2026 diff bps,2026 t,same sign,2026 net pips (rule),2026 t (rule),2026 net pips (always),holds
id,,,,,,,,
AUDJPY-N1/post1,-1.343,-3.326,-4.504,True,-5.791,-13.277,NaN,False
GBPJPY-N1/post1,-2.116,-1.569,-3.166,True,-10.386,-17.876,NaN,False
N-EJ-1/post1,-1.261,-1.749,-4.606,True,-6.396,-14.996,NaN,False
N-UJ-3/post1,-1.163,-1.565,-4.260,True,-2.718,-8.451,NaN,False


holds in 2026: 0 of 4  (by chance about half would show the same sign)


In [8]:
summary = {"study": "FXS10b", "prereg": "discussions/FX_Intraday_Seasonality/006_claude_fxs10b_winning_days_preregistration.md",
           "synthetic": SYNTHETIC, "window": "2020-2025", "candidates_sha256": SHA, "E0": E0.reset_index().to_dict(orient="records"),
           "candidates": cand.round(4).to_dict(orient="records"), "measures": MEAS.round(4).reset_index().to_dict(orient="records") if len(MEAS) else [],
           "holdout_2026": H.round(4).to_dict(orient="records") if len(H) else []}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")

written D:\Projects\equity_trading\notebooks\FX_Intraday_Seasonality\Portfolio\10b\summary.json
